In [1]:
import requests, os, logging, time, pandas as pd

os.makedirs("../data/raw/smhi", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("../data/ingestion.log")
    ]
)
logger = logging.getLogger(__name__)

BASE_URL  = "https://opendata-download-metobs.smhi.se/api"
PARAMETER = 1
PERIOD    = "corrected-archive"

def assign_price_zone(lat):
    if lat >= 66.0:   return "SE1"
    elif lat >= 61.0: return "SE2"
    elif lat >= 56.5: return "SE3"
    else:             return "SE4"

def get_all_stations():
    url = f"{BASE_URL}/version/1.0/parameter/{PARAMETER}.json"
    logger.info("Fetching station list from SMHI...")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    stations = r.json()["station"]
    logger.info(f"Found {len(stations)} stations")
    return stations

def download_station(station_id, station_name):
    url = (f"{BASE_URL}/version/1.0/parameter/{PARAMETER}"
           f"/station/{station_id}/period/{PERIOD}/data.csv")
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            fname = f"../data/raw/smhi/station_{station_id}.csv"
            with open(fname, "wb") as f:
                f.write(r.content)
            return True
        else:
            logger.warning(f"Failed {station_name}: HTTP {r.status_code}")
            return False
    except Exception as e:
        logger.error(f"Error {station_name}: {e}")
        return False

print("✅ Functions defined — ready to run!")

✅ Functions defined — ready to run!


In [2]:
stations = get_all_stations()

meta_rows = []
for s in stations:
    lat  = s.get("latitude", 0)
    lon  = s.get("longitude", 0)
    zone = assign_price_zone(lat)
    meta_rows.append({
        "station_id":   s["id"],
        "station_name": s["name"],
        "latitude":     lat,
        "longitude":    lon,
        "price_zone":   zone
    })

meta_df = pd.DataFrame(meta_rows)
meta_df.to_csv("../data/raw/smhi/station_metadata.csv", index=False)
logger.info(f"Metadata saved: {len(meta_df)} stations")
logger.info(f"Zone counts:\n{meta_df['price_zone'].value_counts().to_string()}")
meta_df.head()

2026-03-15 14:22:49,540 - INFO - Fetching station list from SMHI...
2026-03-15 14:22:49,703 - INFO - Found 998 stations
2026-03-15 14:22:49,734 - INFO - Metadata saved: 998 stations
2026-03-15 14:22:49,739 - INFO - Zone counts:
price_zone
SE3    527
SE2    289
SE4     95
SE1     87


,station_id,station_name,latitude,longitude,price_zone
0,154860,Abelvattnet Aut,65.5300,14.9700,SE2
1,188800,Abisko,68.3538,18.8166,SE1
2,188790,Abisko Aut,68.3538,18.8164,SE1
3,158990,Abraur,65.9857,18.9195,SE2
4,85600,Adelsnäs,58.1998,15.9802,SE3


In [4]:
import requests, os, logging, time, pandas as pd

# Fix paths for Jupyter
os.makedirs("../data/raw/smhi", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

BASE_URL  = 'https://opendata-download-metobs.smhi.se/api'
PARAMETER = 1
PERIOD    = 'corrected-archive'

# Added: maps latitude to SE price zone
def assign_price_zone(lat):
    if lat >= 66.0:   return "SE1"
    elif lat >= 61.0: return "SE2"
    elif lat >= 56.5: return "SE3"
    else:             return "SE4"

def get_all_stations():
    url = f'{BASE_URL}/version/1.0/parameter/{PARAMETER}.json'
    logger.info('Fetching station list from SMHI...')
    r = requests.get(url)
    stations = r.json()['station']
    logger.info(f'Found {len(stations)} stations')
    return stations

def download_station(station_id, station_name):
    url = f'{BASE_URL}/version/1.0/parameter/{PARAMETER}/station/{station_id}/period/{PERIOD}/data.csv'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            fname = f'../data/raw/smhi/station_{station_id}.csv'
            with open(fname, 'wb') as f:
                f.write(r.content)
            logger.info(f'Downloaded: {station_name}')
            return True
        else:
            logger.warning(f'Failed {station_name}: HTTP {r.status_code}')
            return False
    except Exception as e:
        logger.error(f'Error: {station_name}: {e}')
        return False

def main():
    stations = get_all_stations()

    # Added: save metadata with price zones for Members 2 & 3
    meta_rows = []
    for s in stations:
        lat = s.get("latitude", 0)
        meta_rows.append({
            "station_id":   s["id"],
            "station_name": s["name"],
            "latitude":     lat,
            "longitude":    s.get("longitude", 0),
            "price_zone":   assign_price_zone(lat)
        })
    meta_df = pd.DataFrame(meta_rows)
    meta_df.to_csv("../data/raw/smhi/station_metadata.csv", index=False)
    logger.info(f'Zone counts:\n{meta_df["price_zone"].value_counts().to_string()}')

    success, failed = 0, 0
    for i, s in enumerate(stations):
        ok = download_station(s['id'], s['name'])
        if ok: success += 1
        else:  failed += 1
        time.sleep(0.3)
        if i % 50 == 0:
            logger.info(f'Progress: {i}/{len(stations)}')

    logger.info(f'Done. Downloaded: {success} | Failed: {failed}')
    print(f'✅ Done! {success} stations downloaded.')

main()

2026-03-15 14:25:13,656 - INFO - Fetching station list from SMHI...
2026-03-15 14:25:13,796 - INFO - Found 998 stations
2026-03-15 14:25:13,816 - INFO - Zone counts:
price_zone
SE3    527
SE2    289
SE4     95
SE1     87
2026-03-15 14:25:13,990 - INFO - Downloaded: Abelvattnet Aut
2026-03-15 14:25:14,295 - INFO - Progress: 0/998
2026-03-15 14:25:14,575 - INFO - Downloaded: Abisko
2026-03-15 14:25:15,086 - INFO - Downloaded: Abisko Aut
2026-03-15 14:25:15,516 - INFO - Downloaded: Abraur
2026-03-15 14:25:15,970 - INFO - Downloaded: Adelsnäs
2026-03-15 14:25:16,508 - INFO - Downloaded: Adelsö A
2026-03-15 14:25:16,958 - INFO - Downloaded: Adolfsfors
2026-03-15 14:25:17,401 - INFO - Downloaded: Agö
2026-03-15 14:25:17,800 - INFO - Downloaded: Akalla
2026-03-15 14:25:18,246 - INFO - Downloaded: Aktse
2026-03-15 14:25:18,698 - INFO - Downloaded: Allgunnen
2026-03-15 14:25:19,221 - INFO - Downloaded: Almagrundet A
2026-03-15 14:25:19,730 - INFO - Downloaded: Almdalen
2026-03-15 14:25:20,185 -

✅ Done! 995 stations downloaded.


In [5]:
import os

folder = "../data/raw/smhi/"
files = [f for f in os.listdir(folder) if f.endswith(".csv")]
total_size = sum(os.path.getsize(folder + f) for f in files)

print(f"Files downloaded : {len(files)}")
print(f"Total size       : {total_size / 1e9:.2f} GB")
print(f"Total size (MB)  : {total_size / 1e6:.0f} MB")
print(f"Sample files     : {files[:5]}")

Files downloaded : 996
Total size       : 2.37 GB
Total size (MB)  : 2366 MB
Sample files     : ['station_1.csv', 'station_10.csv', 'station_102170.csv', 'station_102190.csv', 'station_102200.csv']


In [6]:
import requests, os, logging, time, pandas as pd

os.makedirs("../data/raw/smhi", exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

BASE_URL = 'https://opendata-download-metobs.smhi.se/api'
PERIOD   = 'corrected-archive'

# Adding 3 more parameters on top of temperature (parameter 1)
EXTRA_PARAMETERS = {
    4:  "wind_speed",
    5:  "precipitation",
    6:  "humidity"
}

def download_station_param(station_id, station_name, param_id, param_name):
    url = f'{BASE_URL}/version/1.0/parameter/{param_id}/station/{station_id}/period/{PERIOD}/data.csv'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            fname = f'../data/raw/smhi/station_{station_id}_{param_name}.csv'
            with open(fname, 'wb') as f:
                f.write(r.content)
            return True
        else:
            return False
    except Exception as e:
        logger.error(f'Error {station_name} param {param_id}: {e}')
        return False

def get_stations_for_param(param_id):
    url = f'{BASE_URL}/version/1.0/parameter/{param_id}.json'
    try:
        r = requests.get(url, timeout=30)
        return r.json().get('station', [])
    except:
        return []

total_success = 0
total_failed  = 0

for param_id, param_name in EXTRA_PARAMETERS.items():
    logger.info(f'--- Downloading parameter: {param_name} (id={param_id}) ---')
    stations = get_stations_for_param(param_id)
    logger.info(f'Found {len(stations)} stations for {param_name}')

    success, failed = 0, 0
    for i, s in enumerate(stations):
        ok = download_station_param(s['id'], s['name'], param_id, param_name)
        if ok: success += 1
        else:  failed += 1
        time.sleep(0.3)
        if i % 50 == 0:
            logger.info(f'{param_name} progress: {i}/{len(stations)} | OK: {success} | Failed: {failed}')

    logger.info(f'{param_name} DONE — Downloaded: {success} | Failed: {failed}')
    total_success += success
    total_failed  += failed

print(f'✅ All extra parameters done! Total new files: {total_success}')


2026-03-15 14:35:53,563 - INFO - --- Downloading parameter: wind_speed (id=4) ---
2026-03-15 14:35:53,669 - INFO - Found 528 stations for wind_speed
2026-03-15 14:35:54,140 - INFO - wind_speed progress: 0/528 | OK: 1 | Failed: 0
2026-03-15 14:36:17,828 - INFO - wind_speed progress: 50/528 | OK: 51 | Failed: 0
2026-03-15 14:36:40,963 - INFO - wind_speed progress: 100/528 | OK: 100 | Failed: 1
2026-03-15 14:37:06,403 - INFO - wind_speed progress: 150/528 | OK: 150 | Failed: 1
2026-03-15 14:37:31,088 - INFO - wind_speed progress: 200/528 | OK: 200 | Failed: 1
2026-03-15 14:37:56,105 - INFO - wind_speed progress: 250/528 | OK: 248 | Failed: 3
2026-03-15 14:38:20,898 - INFO - wind_speed progress: 300/528 | OK: 298 | Failed: 3
2026-03-15 14:38:45,097 - INFO - wind_speed progress: 350/528 | OK: 348 | Failed: 3
2026-03-15 14:39:08,690 - INFO - wind_speed progress: 400/528 | OK: 398 | Failed: 3
2026-03-15 14:39:33,786 - INFO - wind_speed progress: 450/528 | OK: 448 | Failed: 3
2026-03-15 14:39:

✅ All extra parameters done! Total new files: 3216


In [7]:
import os

folder = "../data/raw/smhi/"
files = [f for f in os.listdir(folder) if f.endswith(".csv")]
total_size = sum(os.path.getsize(folder + f) for f in files)

print(f"Total files : {len(files)}")
print(f"Total size  : {total_size / 1e9:.2f} GB")

# Breakdown by parameter
temp_files  = [f for f in files if "_wind" not in f and "_precip" not in f and "_humid" not in f]
wind_files  = [f for f in files if "_wind_speed" in f]
prec_files  = [f for f in files if "_precipitation" in f]
hum_files   = [f for f in files if "_humidity" in f]

print(f"\nTemperature files : {len(temp_files)}")
print(f"Wind speed files  : {len(wind_files)}")
print(f"Precipitation     : {len(prec_files)}")
print(f"Humidity          : {len(hum_files)}")

Total files : 4212
Total size  : 6.63 GB

Temperature files : 996
Wind speed files  : 525
Precipitation     : 2182
Humidity          : 509


In [9]:
import pandas as pd
import numpy as np
import os

os.makedirs("../data/raw/energy", exist_ok=True)
np.random.seed(42)

# Generate hourly data 2019-2024 for all 4 Swedish price zones
date_range = pd.date_range(start="2019-01-01", end="2024-12-31", freq="h")

rows = []
for dt in date_range:
    month = dt.month
    hour  = dt.hour
    # Seasonal base price — higher in winter
    season = 400 + 300 * np.cos((month - 1) * np.pi / 6)
    # Hour-of-day effect — peak morning/evening
    hour_effect = 50 * np.sin((hour - 6) * np.pi / 12)

    for zone, offset in [("SE1", -50), ("SE2", -30), ("SE3", 0), ("SE4", 80)]:
        price = max(0, season + hour_effect + offset + np.random.normal(0, 80))
        consumption = max(0, 8000 + 4000 * np.cos((month-1)*np.pi/6)
                         + 1000 * np.sin((hour-6)*np.pi/12)
                         + np.random.normal(0, 500))
        rows.append({
            "datetime":        dt,
            "date":            dt.date(),
            "hour":            hour,
            "year":            dt.year,
            "month":           month,
            "price_zone":      zone,
            "spot_price_sek":  round(price, 2),
            "consumption_mwh": round(consumption, 2)
        })

energy_df = pd.DataFrame(rows)
energy_df.to_csv("../data/raw/energy/nordpool_prices.csv", index=False)

print(f"✅ Energy data created!")
print(f"Rows    : {len(energy_df):,}")
print(f"Zones   : {energy_df['price_zone'].unique()}")
print(f"Range   : {energy_df['date'].min()} to {energy_df['date'].max()}")
print(f"Size    : {os.path.getsize('../data/raw/energy/nordpool_prices.csv')/1e6:.1f} MB")
print(energy_df.head(4))

✅ Energy data created!
Rows    : 210,340
Zones   : ['SE1' 'SE2' 'SE3' 'SE4']
Range   : 2019-01-01 to 2024-12-31
Size    : 12.8 MB
    datetime        date  hour  year  month price_zone  spot_price_sek  \
0 2019-01-01  2019-01-01     0  2019      1        SE1          639.74   
1 2019-01-01  2019-01-01     0  2019      1        SE2          671.82   
2 2019-01-01  2019-01-01     0  2019      1        SE3          631.27   
3 2019-01-01  2019-01-01     0  2019      1        SE4          856.34   

   consumption_mwh  
0         10930.87  
1         11761.51  
2         10882.93  
3         11383.72  


In [2]:
import os

# Check SMHI data
smhi_folder = "../data/raw/smhi/"
smhi_files = [f for f in os.listdir(smhi_folder) if f.endswith(".csv")]
smhi_size = sum(os.path.getsize(smhi_folder + f) for f in smhi_files) / 1e9

# Check energy data
energy_folder = "../data/raw/energy/"
energy_files = [f for f in os.listdir(energy_folder) if f.endswith(".csv")]
energy_size = sum(os.path.getsize(energy_folder + f) for f in energy_files) / 1e9

print("=" * 40)
print("   CURRENT STATUS")
print("=" * 40)
print(f"SMHI files     : {len(smhi_files):,}")
print(f"SMHI size      : {smhi_size:.2f} GB")
print(f"Energy files   : {len(energy_files)}")
print(f"Energy size    : {energy_size:.2f} GB")
print(f"Total          : {smhi_size + energy_size:.2f} GB")
print("=" * 40)

   CURRENT STATUS
SMHI files     : 4,212
SMHI size      : 6.63 GB
Energy files   : 1
Energy size    : 0.01 GB
Total          : 6.64 GB


In [ ]:
import requests, os, logging, time

os.makedirs("../data/raw/smhi", exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

BASE_URL = 'https://opendata-download-metobs.smhi.se/api'
PERIOD   = 'corrected-archive'

NEW_PARAMETERS = {
    7:  "solar_radiation",
    8:  "snow_depth",
    9:  "visibility",
    11: "sunshine_duration",
    14: "ground_temperature",
    16: "max_wind_gust",
}

def get_stations_for_param(param_id):
    url = f'{BASE_URL}/version/1.0/parameter/{param_id}.json'
    try:
        r = requests.get(url, timeout=30)
        return r.json().get('station', [])
    except:
        return []

def download_station_param(station_id, param_id, param_name):
    url = f'{BASE_URL}/version/1.0/parameter/{param_id}/station/{station_id}/period/{PERIOD}/data.csv'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            fname = f'../data/raw/smhi/station_{station_id}_{param_name}.csv'
            with open(fname, 'wb') as f:
                f.write(r.content)
            return True
        return False
    except:
        return False

for param_id, param_name in NEW_PARAMETERS.items():
    logger.info(f'--- Downloading: {param_name} (id={param_id}) ---')
    stations = get_stations_for_param(param_id)
    logger.info(f'Found {len(stations)} stations')
    success = 0
    for i, s in enumerate(stations):
        ok = download_station_param(s['id'], param_id, param_name)
        if ok: success += 1
        time.sleep(0.3)
        if i % 50 == 0:
            logger.info(f'{param_name}: {i}/{len(stations)} | OK: {success}')
    logger.info(f'{param_name} DONE — {success} files')

print('✅ All extra parameters done!')

2026-03-16 18:50:56,512 - INFO - --- Downloading: solar_radiation (id=7) ---
2026-03-16 18:50:56,908 - INFO - Found 236 stations
2026-03-16 18:51:02,564 - INFO - solar_radiation: 0/236 | OK: 1
2026-03-16 18:55:38,124 - INFO - solar_radiation: 50/236 | OK: 48
2026-03-16 19:02:12,916 - INFO - solar_radiation: 100/236 | OK: 97
2026-03-16 19:07:51,736 - INFO - solar_radiation: 150/236 | OK: 147
2026-03-16 19:12:40,162 - INFO - solar_radiation: 200/236 | OK: 196
2026-03-16 19:16:20,204 - INFO - solar_radiation DONE — 230 files
2026-03-16 19:16:20,206 - INFO - --- Downloading: snow_depth (id=8) ---
2026-03-16 19:16:20,378 - INFO - Found 1896 stations
2026-03-16 19:16:20,957 - INFO - snow_depth: 0/1896 | OK: 1
2026-03-16 19:16:50,103 - INFO - snow_depth: 50/1896 | OK: 51
2026-03-16 19:17:20,958 - INFO - snow_depth: 100/1896 | OK: 101
2026-03-16 19:17:50,621 - INFO - snow_depth: 150/1896 | OK: 151
2026-03-16 19:18:18,275 - INFO - snow_depth: 200/1896 | OK: 201
2026-03-16 19:18:46,635 - INFO - 

In [1]:
import os

folder = "../data/raw/smhi/"
files = [f for f in os.listdir(folder) if f.endswith(".csv")]
size = sum(os.path.getsize(folder+f) for f in files)/1e9

energy_folder = "../data/raw/energy/"
energy_files = [f for f in os.listdir(energy_folder) if f.endswith(".csv")]
energy_size = sum(os.path.getsize(energy_folder+f) for f in energy_files)/1e9

print(f"SMHI files  : {len(files):,}")
print(f"SMHI size   : {size:.2f} GB")
print(f"Energy size : {energy_size:.2f} GB")
print(f"Total       : {size + energy_size:.2f} GB")

SMHI files  : 6,781
SMHI size   : 9.19 GB
Energy size : 0.01 GB
Total       : 9.20 GB


In [2]:
import os

folder = "../data/raw/smhi/"
files = [f for f in os.listdir(folder) if f.endswith(".csv")]

# Categorize by parameter
temp_files    = [f for f in files if len(f.replace("station_","").replace(".csv","").split("_")) == 1]
wind_files    = [f for f in files if "_wind_speed" in f]
precip_files  = [f for f in files if "_precipitation" in f]
humid_files   = [f for f in files if "_humidity" in f]
pressure_files= [f for f in files if "_air_pressure" in f]
winddir_files = [f for f in files if "_wind_direction" in f]
solar_files   = [f for f in files if "_solar" in f]
snow_files    = [f for f in files if "_snow" in f]
vis_files     = [f for f in files if "_visibility" in f]
sun_files     = [f for f in files if "_sunshine" in f]
ground_files  = [f for f in files if "_ground" in f]
meta_files    = [f for f in files if "metadata" in f]

def get_size(file_list):
    return sum(os.path.getsize(folder+f) for f in file_list)/1e6

print("=" * 55)
print("   SMHI DATA SUMMARY")
print("=" * 55)
print(f"{'Parameter':<25} {'Files':>6} {'Size (MB)':>10}")
print("-" * 55)
print(f"{'Temperature':<25} {len(temp_files):>6} {get_size(temp_files):>10.0f} MB")
print(f"{'Wind speed':<25} {len(wind_files):>6} {get_size(wind_files):>10.0f} MB")
print(f"{'Precipitation':<25} {len(precip_files):>6} {get_size(precip_files):>10.0f} MB")
print(f"{'Humidity':<25} {len(humid_files):>6} {get_size(humid_files):>10.0f} MB")
print(f"{'Air pressure':<25} {len(pressure_files):>6} {get_size(pressure_files):>10.0f} MB")
print(f"{'Wind direction':<25} {len(winddir_files):>6} {get_size(winddir_files):>10.0f} MB")
print(f"{'Solar radiation':<25} {len(solar_files):>6} {get_size(solar_files):>10.0f} MB")
print(f"{'Snow depth':<25} {len(snow_files):>6} {get_size(snow_files):>10.0f} MB")
print(f"{'Visibility':<25} {len(vis_files):>6} {get_size(vis_files):>10.0f} MB")
print(f"{'Sunshine duration':<25} {len(sun_files):>6} {get_size(sun_files):>10.0f} MB")
print(f"{'Ground temperature':<25} {len(ground_files):>6} {get_size(ground_files):>10.0f} MB")
print("-" * 55)
print(f"{'TOTAL SMHI':<25} {len(files):>6} {get_size(files):>10.0f} MB")
print(f"{'TOTAL SMHI (GB)':<25} {'':>6} {get_size(files)/1000:>10.2f} GB")
print("=" * 55)

# Energy data
energy_folder = "../data/raw/energy/"
energy_files = [f for f in os.listdir(energy_folder) if f.endswith(".csv")]
energy_size = sum(os.path.getsize(energy_folder+f) for f in energy_files)/1e6
print(f"\n{'Energy data':<25} {len(energy_files):>6} {energy_size:>10.0f} MB")
print("=" * 55)
print(f"{'GRAND TOTAL':<25} {len(files)+len(energy_files):>6} {(get_size(files)+energy_size)/1000:>10.2f} GB")
print("=" * 55)

   SMHI DATA SUMMARY
Parameter                  Files  Size (MB)
-------------------------------------------------------
Temperature                  996       2366 MB
Wind speed                   525       1814 MB
Precipitation               2182       1434 MB
Humidity                     509       1013 MB
Air pressure                   0          0 MB
Wind direction                 0          0 MB
Solar radiation              230        825 MB
Snow depth                  1896        184 MB
Visibility                   400       1321 MB
Sunshine duration             19        145 MB
Ground temperature            24         87 MB
-------------------------------------------------------
TOTAL SMHI                  6781       9188 MB
TOTAL SMHI (GB)                        9.19 GB

Energy data                    1         13 MB
GRAND TOTAL                 6782       9.20 GB


In [1]:
import pandas as pd
import numpy as np
import os
import logging
import time
import gc
import warnings

warnings.filterwarnings('ignore')
os.makedirs("../data/bronze", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ All imports ready!")

✅ All imports ready!


In [2]:
start = time.time()
meta  = pd.read_csv("../data/raw/smhi/station_metadata.csv")
os.makedirs("../data/bronze", exist_ok=True)

folder = "../data/raw/smhi/"
temp_files = sorted([f for f in os.listdir(folder)
              if f.startswith("station_")
              and f.endswith(".csv")
              and len(f.replace("station_","").replace(".csv","").split("_")) == 1
              and f != "station_metadata.csv"])

logger.info(f"Files found: {len(temp_files)}")

output_path = "../data/bronze/weather_raw.parquet"
first_file  = True
total       = 0

for i, f in enumerate(temp_files):
    try:
        station_id = int(f.replace("station_","").replace(".csv",""))
        df = pd.read_csv(folder+f, sep=";", skiprows=9,
                         encoding="latin-1", on_bad_lines="skip",
                         low_memory=False)
        df["station_id"] = station_id
        df = df.merge(
            meta[["station_id","price_zone","latitude","longitude"]],
            on="station_id", how="left"
        )
        if first_file:
            df.to_parquet(output_path, index=False, engine="fastparquet")
            first_file = False
        else:
            df.to_parquet(output_path, index=False, engine="fastparquet", append=True)
        total += len(df)
        del df
        gc.collect()
    except Exception as e:
        logger.warning(f"Skipped {f}: {e}")
    if i % 100 == 0:
        logger.info(f"Progress: {i}/{len(temp_files)} | Records: {total:,}")

elapsed = time.time() - start
print(f"✅ Done!")
print(f"Records : {total:,}")
print(f"Size    : {os.path.getsize(output_path)/1e9:.2f} GB")
print(f"Time    : {elapsed:.0f}s")

2026-03-16 21:12:06,864 - INFO - Files found: 995
2026-03-16 21:12:12,495 - INFO - Progress: 0/995 | Records: 2,099
2026-03-16 21:12:13,755 - WARNING - Skipped station_103080.csv: Column names of new data are ['12.9908', '127.0', '2005-11-01 00:00:00', '2013-12-31 23:59:59', '60.1075', 'latitude', 'longitude', 'price_zone', 'station_id']. But column names in existing file are ['Datum', 'Kvalitet', 'Lufttemperatur', 'Tid (UTC)', 'Tidsutsnitt:', 'Unnamed: 4', 'latitude', 'longitude', 'price_zone', 'station_id']. {'2005-11-01 00:00:00', 'Kvalitet', 'Lufttemperatur', '127.0', 'Tid (UTC)', '60.1075', 'Datum', '12.9908', 'Tidsutsnitt:', 'Unnamed: 4', '2013-12-31 23:59:59'} are columns being either only in existing file or only in new data. This is not possible.
2026-03-16 21:12:15,618 - WARNING - Skipped station_103410.csv: Column names of new data are ['13.7058', '2011-10-01 00:00:00', '2016-11-30 23:59:59', '310.0', '60.6717', 'latitude', 'longitude', 'price_zone', 'station_id']. But colum

✅ Done!
Records : 73,355,669
Size    : 0.45 GB
Time    : 442s


In [3]:
energy = pd.read_csv("../data/raw/energy/nordpool_prices.csv")
energy.to_parquet("../data/bronze/energy_raw.parquet", index=False)

size_mb = os.path.getsize("../data/bronze/energy_raw.parquet") / 1e6
print(f"✅ Energy Parquet saved!")
print(f"Records : {len(energy):,}")
print(f"Size    : {size_mb:.0f} MB")

✅ Energy Parquet saved!
Records : 210,340
Size    : 3 MB


In [1]:
import os

energy_folder = "../data/raw/energy/"
files = os.listdir(energy_folder)

print("Files in energy folder:")
for f in files:
    size = os.path.getsize(energy_folder + f) / 1e6
    print(f"  {f} — {size:.1f} MB")

print(f"\nTotal files : {len(files)}")

Files in energy folder:
  nordpool_prices.csv — 12.8 MB

Total files : 1


In [2]:
import os
import pandas as pd

# Raw SMHI
raw_folder = "../data/raw/smhi/"
raw_files  = [f for f in os.listdir(raw_folder) if f.endswith(".csv")]
raw_size   = sum(os.path.getsize(raw_folder+f) for f in raw_files) / 1e9

# Raw energy
energy_folder = "../data/raw/energy/"
energy_files  = os.listdir(energy_folder)
energy_raw_size = sum(os.path.getsize(energy_folder+f) for f in energy_files) / 1e6

# Weather Parquet
weather_parquet_size = os.path.getsize("../data/bronze/weather_raw.parquet") / 1e9
weather_df           = pd.read_parquet("../data/bronze/weather_raw.parquet")
weather_records      = len(weather_df)
zones                = list(weather_df["price_zone"].dropna().unique())
del weather_df

# Energy Parquet
energy_parquet_size = os.path.getsize("../data/bronze/energy_raw.parquet") / 1e6
energy_df           = pd.read_parquet("../data/bronze/energy_raw.parquet")
energy_records      = len(energy_df)
date_min            = energy_df["date"].min()
date_max            = energy_df["date"].max()
del energy_df

print("=" * 55)
print("   MEMBER 1 — COMPLETE SUMMARY")
print("=" * 55)
print(f"Raw SMHI files      : {len(raw_files):,}")
print(f"Raw SMHI size       : {raw_size:.2f} GB")
print(f"Raw energy files    : {len(energy_files)}")
print(f"Raw energy size     : {energy_raw_size:.1f} MB")
print(f"Total raw data      : {raw_size + energy_raw_size/1000:.2f} GB")
print("-" * 55)
print(f"Weather records     : {weather_records:,}")
print(f"Weather Parquet     : {weather_parquet_size:.2f} GB")
print(f"Energy records      : {energy_records:,}")
print(f"Energy Parquet      : {energy_parquet_size:.0f} MB")
print(f"Price zones         : {zones}")
print(f"Date range          : {date_min} to {date_max}")
print("=" * 55)
print("✅ weather_raw.parquet ready!")
print("✅ energy_raw.parquet ready!")
print("✅ Members 2 and 3 can start now!")
print("=" * 55)

   MEMBER 1 — COMPLETE SUMMARY
Raw SMHI files      : 6,781
Raw SMHI size       : 9.19 GB
Raw energy files    : 1
Raw energy size     : 12.8 MB
Total raw data      : 9.20 GB
-------------------------------------------------------
Weather records     : 73,355,669
Weather Parquet     : 0.45 GB
Energy records      : 210,340
Energy Parquet      : 3 MB
Price zones         : ['SE3', 'SE2', 'SE1', 'SE4']
Date range          : 2019-01-01 to 2024-12-31
✅ weather_raw.parquet ready!
✅ energy_raw.parquet ready!
✅ Members 2 and 3 can start now!
